In [ ]:
import pandas as pd
df = pd.read_json('bugzilla_2020_2026_200k.json')

In [7]:
print(type(df))
print(pd.__version__)

<class 'pandas.DataFrame'>
3.0.1


In [8]:
import pandas as pd
df = pd.read_json('bugzilla_2020_2026_200k.json')
df_clean = df.astype(str)
duplicates = df_clean[df_clean.duplicated(keep=False)]
print(len(duplicates))

74465


In [9]:
import pandas as pd

# Load data
df = pd.read_json('bugzilla_2020_2026_200k.json')

# Step 1: Convert to string (to make rows hashable)
df_clean = df.astype(str)

# Step 2: Remove exact duplicate rows
df_clean = df_clean.drop_duplicates()

print("After removing exact duplicates:", len(df_clean))

# Step 3: Group by ID and count occurrences
id_counts = df_clean.groupby("id").size().reset_index(name="count")

# Step 4: Show IDs that still repeat
duplicates_by_id = id_counts[id_counts["count"] > 1]

print("\nIDs with multiple entries:")
print(duplicates_by_id)

print("\nTotal duplicate IDs:", len(duplicates_by_id))

After removing exact duplicates: 160905

IDs with multiple entries:
             id  count
89913   2253539      2
119372  2346806      2
153824  2433906      2
157789  2444095      2
158992  2447786      2
159832  2450091      3
160017  2450708      2
160270  2451396      2
160779  2452799      2
160787  2452809      2

Total duplicate IDs: 10


In [1]:
import pandas as pd

# Load data
df = pd.read_json("bugzilla_2020_2026_200k.json")

print("Original rows:", len(df))

# Convert to string so exact row duplicates are detected consistently
df_clean = df.astype(str)

# Exact duplicate rows
exact_dup_rows = df_clean.duplicated(keep=False).sum()
print("Exact duplicate rows (keep=False):", exact_dup_rows)

# Unique duplicate row groups (one count per duplicate row after first occurrence)
exact_dup_row_groups = df_clean.duplicated().sum()
print("Exact duplicate row groups:", exact_dup_row_groups)

# Remove exact duplicate rows
df_unique = df_clean.drop_duplicates()
print("Rows after drop_duplicates():", len(df_unique))

# Count repeated IDs after removing exact duplicate rows
id_counts = df_unique.groupby("id").size().reset_index(name="count")
dup_ids = id_counts[id_counts["count"] > 1]

print("IDs with multiple rows after exact duplicate removal:", len(dup_ids))
print("Total rows for those duplicate IDs:", dup_ids["count"].sum())

# Optional: compare with original duplicate IDs before cleaning
orig_id_counts = df.groupby("id").size().reset_index(name="count")
orig_dup_ids = orig_id_counts[orig_id_counts["count"] > 1]
print("Original IDs with multiple rows:", len(orig_dup_ids))
print("Original rows for duplicate IDs:", orig_dup_ids["count"].sum())

# Inspect a few duplicate IDs
print("\nSample duplicate IDs after cleaning:")
print(dup_ids.head(20).to_string(index=False))


Original rows: 200000
Exact duplicate rows (keep=False): 74465
Exact duplicate row groups: 39095
Rows after drop_duplicates(): 160905
IDs with multiple rows after exact duplicate removal: 10
Total rows for those duplicate IDs: 21
Original IDs with multiple rows: 35376
Original rows for duplicate IDs: 74482

Sample duplicate IDs after cleaning:
     id  count
2253539      2
2346806      2
2433906      2
2444095      2
2447786      2
2450091      3
2450708      2
2451396      2
2452799      2
2452809      2


In [3]:
df_unique.to_csv("bugs_data.csv", index =False)

In [5]:
from concurrent.futures import ThreadPoolExecutor
import requests
import pandas as pd
import json

INPUT_FILE = "bugs_data.csv"
OUTPUT_FILE = "bug_history.json"

df = pd.read_csv(INPUT_FILE)
df["id"] = df["id"].astype(str)
bug_ids = df["id"].unique().tolist()

def fetch_history(bug_id):
    url = f"https://bugzilla.redhat.com/rest/bug/{bug_id}/history"
    try:
        res = requests.get(url, timeout=10)
        if res.status_code == 200:
            return res.json()
    except:
        return None

all_histories = []

with ThreadPoolExecutor(max_workers=20) as executor:
    results = executor.map(fetch_history, bug_ids)

    for i, result in enumerate(results):
        if result:
            all_histories.append(result)

        if i % 1000 == 0:
            print(f"Processed {i}/{len(bug_ids)}")

with open(OUTPUT_FILE, "w") as f:
    json.dump(all_histories, f)

print("✅ Done")

Processed 0/160894
Processed 1000/160894
Processed 2000/160894
Processed 3000/160894
Processed 4000/160894
Processed 5000/160894
Processed 6000/160894
Processed 7000/160894
Processed 8000/160894
Processed 9000/160894
Processed 10000/160894
Processed 11000/160894
Processed 12000/160894
Processed 13000/160894
Processed 14000/160894
Processed 15000/160894
Processed 16000/160894
Processed 17000/160894
Processed 18000/160894
Processed 19000/160894
Processed 20000/160894
Processed 21000/160894
Processed 22000/160894
Processed 23000/160894
Processed 24000/160894
Processed 25000/160894
Processed 26000/160894
Processed 27000/160894
Processed 28000/160894
Processed 29000/160894
Processed 30000/160894
Processed 31000/160894
Processed 32000/160894
Processed 33000/160894
Processed 34000/160894
Processed 35000/160894
Processed 36000/160894
Processed 37000/160894
Processed 38000/160894
Processed 39000/160894
Processed 40000/160894
Processed 41000/160894
Processed 42000/160894
Processed 43000/160894
P

In [6]:
import json
import pandas as pd

# =========================
# FILES
# =========================
HISTORY_FILE = "bug_history.json"
BASE_FILE = "bugs_data.csv"
OUTPUT_FILE = "final_lifecycle_dataset.csv"

# =========================
# STEP 1: LOAD DATA
# =========================
with open(HISTORY_FILE, "r") as f:
    history_data = json.load(f)

df_base = pd.read_csv(BASE_FILE, low_memory=False)
df_base["id"] = df_base["id"].astype(str)

print("Base rows:", len(df_base))

# =========================
# STEP 2: EXTRACT EVENTS (FAST + SAFE)
# =========================
rows = []

for bug in history_data:
    try:
        bug_info = bug.get("bugs", [{}])[0]
        bug_id = str(bug_info.get("id"))

        for entry in bug_info.get("history", []):
            time = entry.get("when")

            for change in entry.get("changes", []):
                field = change.get("field_name")

                # Track multiple lifecycle fields (BEST PRACTICE)
                if field in ["status", "assigned_to", "priority", "severity"]:
                    rows.append({
                        "id": bug_id,
                        "time": time,
                        "field": field,
                        "value": change.get("added")
                    })

    except Exception:
        continue

# Create DataFrame
df_events = pd.DataFrame(rows)

print("Total events extracted:", len(df_events))

# =========================
# STEP 3: CLEAN + SORT
# =========================
df_events.dropna(subset=["time"], inplace=True)
df_events["time"] = pd.to_datetime(df_events["time"], errors="coerce")
df_events.dropna(subset=["time"], inplace=True)

df_events = df_events.sort_values(["id", "time"])

# =========================
# STEP 4: SPLIT FIELDS (WIDE FORMAT)
# =========================
df_pivot = df_events.pivot_table(
    index=["id", "time"],
    columns="field",
    values="value",
    aggfunc="last"
).reset_index()

print("After pivot:", df_pivot.shape)

# =========================
# STEP 5: MERGE WITH BASE DATA
# =========================
df_final = df_pivot.merge(df_base, on="id", how="left")

print("Final dataset shape:", df_final.shape)

# =========================
# STEP 6: SORT FINAL DATA
# =========================
df_final = df_final.sort_values(["id", "time"])

# =========================
# STEP 7: SAVE
# =========================
df_final.to_csv(OUTPUT_FILE, index=False)

print("✅ Final dataset saved:", OUTPUT_FILE)

Base rows: 160905
Total events extracted: 453625
After pivot: (410568, 6)
Final dataset shape: (410586, 13)
✅ Final dataset saved: final_lifecycle_dataset.csv


In [1]:
import pandas as pd

# =========================
# 1. LOAD DATA
# =========================
df = pd.read_csv("final_lifecycle_dataset.csv")

# =========================
# 2. CLEAN + PREPROCESS
# =========================
# Ensure correct types
df["bug_id"] = df["id"].astype(str)
df["time"] = pd.to_datetime(df["time"], errors="coerce")

# Handle status column (combine if needed)
if "status_x" in df.columns and "status_y" in df.columns:
    df["status"] = df["status_x"].fillna(df["status_y"])
elif "status_x" in df.columns:
    df["status"] = df["status_x"]
elif "status_y" in df.columns:
    df["status"] = df["status_y"]
else:
    raise ValueError("No status column found!")

# Sort by bug_id and time
df = df.sort_values(["bug_id", "time"])

# =========================
# 3. GROUP BY bug_id
# =========================
grouped = df.groupby("bug_id")

# =========================
# 4. FINAL STATE (last record)
# =========================
final_df = grouped.last().reset_index()

# =========================
# 5. STATUS HISTORY
# =========================
lifecycle_df = grouped["status"].apply(list).reset_index()
lifecycle_df.rename(columns={"status": "status_history"}, inplace=True)

# =========================
# 6. TIME FEATURES
# =========================
time_df = grouped["time"].agg(["min", "max"]).reset_index()
time_df.rename(columns={"min": "created_time", "max": "last_update"}, inplace=True)

# Resolution time
time_df["resolution_time"] = time_df["last_update"] - time_df["created_time"]

# =========================
# 7. NUMBER OF UPDATES
# =========================
updates_df = grouped.size().reset_index(name="num_updates")

# =========================
# 8. MERGE EVERYTHING
# =========================
final_dataset = final_df.merge(lifecycle_df, on="bug_id")
final_dataset = final_dataset.merge(time_df, on="bug_id")
final_dataset = final_dataset.merge(updates_df, on="bug_id")

# =========================
# 9. OUTPUT
# =========================
print("Final dataset shape:", final_dataset.shape)
display(final_dataset.head())

# Optional: Save
# final_dataset.to_csv("grouped_bug_data.csv", index=False)

Final dataset shape: (145022, 20)


,bug_id,id,time,assigned_to,priority,severity_x,status_x,product,severity_y,creation_time,data_category,status_y,summary,component,status,status_history,created_time,last_update,resolution_time,num_updates
0,1787185,1787185,2020-01-02 12:04:59+00:00,bpeterse@redhat.com,NaN,NaN,CLOSED,OpenShift Container Platform,medium,2020-01-01 02:52:17+00:00,Public,CLOSED,When clicked on Configmap from webui it shows ...,['Management Console'],CLOSED,"[CLOSED, CLOSED]",2020-01-02 02:27:08+00:00,2020-01-02 12:04:59+00:00,0 days 09:37:51,2
1,1787186,1787186,2020-01-28 08:29:41+00:00,skaplons@redhat.com,NaN,urgent,CLOSED,Red Hat OpenStack,urgent,2020-01-01 05:04:47+00:00,Public,CLOSED,"Undercloud installation stuck at ""TASK [Start ...",['rhosp-director'],CLOSED,"[ASSIGNED, CLOSED, CLOSED]",2020-01-13 11:22:33+00:00,2020-01-28 08:29:41+00:00,14 days 21:07:08,3
2,1787192,1787192,2022-05-26 17:22:44+00:00,vjuranek@redhat.com,high,NaN,CLOSED,Red Hat Enterprise Virtualization Manager,high,2020-01-01 07:15:51+00:00,Public,CLOSED,Host fails to activate in RHV and goes to non-...,['vdsm'],CLOSED,"[CLOSED, POST, CLOSED, CLOSED, CLOSED, NEW, AS...",2020-01-13 15:13:57+00:00,2022-05-26 17:22:44+00:00,864 days 02:08:47,13
3,1787194,1787194,2021-11-16 07:49:56+00:00,lvivier@redhat.com,medium,medium,CLOSED,Red Hat Enterprise Linux Advanced Virtualization,medium,2020-01-01 09:42:35+00:00,Public,CLOSED,After canceling the migration of a vm with VF ...,['qemu-kvm'],CLOSED,"[CLOSED, ASSIGNED, CLOSED, CLOSED, CLOSED, POS...",2020-01-02 19:37:52+00:00,2021-11-16 07:49:56+00:00,683 days 12:12:04,11
4,1787197,1787197,2020-02-10 10:13:37+00:00,jhnidek@redhat.com,high,high,CLOSED,Red Hat Enterprise Linux 8,high,2020-01-01 11:05:51+00:00,Public,CLOSED,rhsmcertd-worker fires as many RHSM queries as...,['subscription-manager'],CLOSED,"[CLOSED, CLOSED, ASSIGNED, CLOSED]",2020-01-10 13:17:22+00:00,2020-02-10 10:13:37+00:00,30 days 20:56:15,4


In [10]:
len(df)

200000

In [4]:
df.sample(10)

,product,severity,id,creation_time,data_category,status,summary,component
1234,Security Response,high,2418872,2025-12-04 17:02:22+00:00,Public,NEW,CVE-2025-40248 kernel: Linux kernel: vsock vul...,[vulnerability]
198130,Fedora,high,2251082,2023-11-22 20:00:12+00:00,Public,CLOSED,Yubikey Manager GUI is unable to detect Yubike...,[yubikey-manager-qt]
150254,Security Response,unspecified,2451268,2026-03-25 11:08:01+00:00,Public,NEW,CVE-2026-23339 kernel: nfc: nci: free skb on n...,[vulnerability]
134953,Fedora,unspecified,2331204,2024-12-09 18:42:06+00:00,Public,CLOSED,moby-engine-27.4.1 is available,[moby-engine]
29619,Fedora,medium,2277068,2024-04-25 05:05:02+00:00,Public,CLOSED,TRIAGE CVE-2024-27280 rubygem-net-http-persist...,[rubygem-net-http-persistent]
124621,Fedora,medium,2252440,2023-12-01 17:08:02+00:00,Public,NEW,glibc: Investigate removal of glibc-cs-path.patch,[glibc]
150948,Fedora,unspecified,1799466,2020-02-06 17:16:06+00:00,Public,CLOSED,gpsdrive: FTBFS in Fedora rawhide/f32,[gpsdrive]
38541,Virtualization Tools,unspecified,2349762,2025-03-04 11:59:52+00:00,Public,CLOSED,virt-v2v: error: inspection could not detect t...,[libguestfs]
9361,Fedora,unspecified,2346003,2025-02-16 16:09:05+00:00,Public,CLOSED,gram_grep-0.9.2 is available,[gram_grep]
156358,Red Hat OpenStack,high,1898633,2020-11-17 17:06:09+00:00,Public,CLOSED,[Docs] Validations in FFU 13->16.1: Not possib...,[documentation]
